# 🎓 Self-Contained Google Colab Training Pipeline

**Goal:** Run the entire data cleaning, feature engineering, pipeline preparation, and hyperparameter tuning in Google Colab to leverage cloud CPU/GPU/TPU and save time from training models locally.

**Outputs:**
- Trained regression and classification models (`.pkl`)
- Best regression & classification pipelines (`best_*.pkl`)
- Model metadata (`model_metadata.json`)
- A downloadable `.zip` package containing all saved artifacts.

## 1. Upload Raw Dataset
Run the cell below to prompt and upload your raw dataset: `ai_student_impact_dataset.csv` (or the copy named `ai_student_impact_dataset (1).csv`).

In [ ]:
from google.colab import files
import os

print("Please upload the raw CSV dataset:")
uploaded = files.upload()

# Locate the uploaded file path
raw_path = list(uploaded.keys())[0]
print(f"Uploaded successfully: {raw_path}")

## 2. Setup Libraries and Directories

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from pathlib import Path
from time import time

# Preprocessing & Pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier

# Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score

# Create directories for model output persistence
os.makedirs("models/saved/regression", exist_ok=True)
os.makedirs("models/saved/classification", exist_ok=True)

print("Setup complete ✓")

## 3. Configuration & Constants

In [ ]:
YEAR_ORDER = {"Freshman": 1, "Sophomore": 2, "Junior": 3, "Senior": 4, "Graduate": 5}
SKILL_ORDER = {"Beginner": 1, "Intermediate": 2, "Advanced": 3}
USE_CASE_GPA_RANK = {
    "Direct_Answer_Generation": 1,
    "Copywriting/Drafting": 2,
    "Ideation": 3,
    "Summarizing_Reading": 4,
    "Debugging/Troubleshooting": 5,
}

CFG = {
    "features": {
        "numerical": [
            "Pre_Semester_GPA", "Weekly_GenAI_Hours", "Tool_Diversity", "Traditional_Study_Hours", 
            "Perceived_AI_Dependency", "Anxiety_Level_During_Exams", "Skill_Retention_Score", 
            "study_ratio", "total_study_hours", "genai_dependency_score", "ai_efficiency", "anxiety_gpa_pressure"
        ],
        "ordinal": [
            "Year_of_Study_enc", "Prompt_Engineering_Skill_enc", "Primary_Use_Case_enc"
        ],
        "categorical": [
            "Major_Category", "Institutional_Policy"
        ],
        "binary": [
            "Paid_Subscription"
        ]
    },
    "models": {
        "regression": ["linear_regression", "random_forest_regressor", "gradient_boosting_regressor"],
        "classification": ["logistic_regression", "random_forest_classifier", "gradient_boosting_classifier"]
    },
    "tuning": {
        "method": "grid_search",
        "cv_folds": 5,
        "scoring_regression": "neg_root_mean_squared_error",
        "scoring_classification": "f1_weighted"
    }
}

SEED = 42
print("Configs loaded ✓")

## 4. Helper Modules (Cleaning, Feature Engineering, Pipelines)

In [ ]:
def clean(df, drop_cols=["Student_ID"]):
    df = df.copy()
    df = df.drop(columns=drop_cols, errors="ignore")
    if "Paid_Subscription" in df.columns:
        df["Paid_Subscription"] = df["Paid_Subscription"].map(
            {True: 1, False: 0, "True": 1, "False": 0, 1.0: 1, 0.0: 0}
        ).fillna(0).astype(int)
    df = df.dropna()
    return df

def engineer_features(df):
    df = df.copy()
    
    # Derived Interaction terms
    df["study_ratio"] = df["Traditional_Study_Hours"] / (df["Weekly_GenAI_Hours"].fillna(0) + 1)
    df["total_study_hours"] = df["Traditional_Study_Hours"] + df["Weekly_GenAI_Hours"]
    df["genai_dependency_score"] = df["Weekly_GenAI_Hours"] * df["Perceived_AI_Dependency"]
    df["ai_efficiency"] = df["Skill_Retention_Score"] / (df["Weekly_GenAI_Hours"] + 1)
    df["anxiety_gpa_pressure"] = df["Anxiety_Level_During_Exams"] * (4 - df["Pre_Semester_GPA"])
    
    # Encodings
    df["Year_of_Study_enc"] = df["Year_of_Study"].map(YEAR_ORDER).fillna(1).astype(int)
    df["Prompt_Engineering_Skill_enc"] = df["Prompt_Engineering_Skill"].map(SKILL_ORDER).fillna(1).astype(int)
    df["Primary_Use_Case_enc"] = df["Primary_Use_Case"].map(USE_CASE_GPA_RANK).fillna(1).astype(int)
    
    # Paid Subscription casting
    df["Paid_Subscription"] = df["Paid_Subscription"].map(
        {True: 1, False: 0, "True": 1, "False": 0, 1.0: 1, 0.0: 0, 1: 1, 0: 0}
    ).fillna(0).astype(int)
    return df

def split(df, target, test_size=0.2, random_state=42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def build_preprocessor():
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    ord_pipeline = Pipeline([
        ("scaler", StandardScaler()),
    ])
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", num_pipeline, CFG["features"]["numerical"]),
        ("ord", ord_pipeline, CFG["features"]["ordinal"]),
        ("cat", cat_pipeline, CFG["features"]["categorical"]),
        ("bin", "passthrough", CFG["features"]["binary"]),
    ])

print("Processing blocks defined ✓")

## 5. Parameter Search Grids Definition

In [ ]:
REGRESSION_MODELS = {
    "linear_regression": LinearRegression(),
    "random_forest_regressor": RandomForestRegressor(n_estimators=100, random_state=SEED),
    "gradient_boosting_regressor": GradientBoostingRegressor(n_estimators=100, random_state=SEED),
}

CLASSIFICATION_MODELS = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "random_forest_classifier": RandomForestClassifier(n_estimators=100, random_state=SEED),
    "gradient_boosting_classifier": GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}

REGRESSION_GRIDS = {
    "linear_regression": {},
    "random_forest_regressor": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [10, 15, None],
    },
    "gradient_boosting_regressor": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 4],
    },
}

CLASSIFICATION_GRIDS = {
    "logistic_regression": {
        "model__C": [0.01, 0.1, 1.0, 10.0],
    },
    "random_forest_classifier": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [5, 10, None],
    },
    "gradient_boosting_classifier": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 4],
    },
}

print("Grids initialized ✓")

## 6. Execute Regression Model Training & Tuning

In [ ]:
# Load and clean
raw_df = load_raw(raw_path)
cleaned_df = clean(raw_df)
engineered_df = engineer_features(cleaned_df)

# Split dataset
X_train_reg, X_test_reg, y_reg_train, y_reg_test = split(engineered_df, "Post_Semester_GPA")
preprocessor = build_preprocessor()

cv_folds = CFG["tuning"]["cv_folds"]
scoring = CFG["tuning"]["scoring_regression"]

reg_fitted = {}
reg_scores = []

for name in CFG["models"]["regression"]:
    pipe = Pipeline([("preprocessor", preprocessor), ("model", REGRESSION_MODELS[name])])
    grid = REGRESSION_GRIDS.get(name, {})
    
    if grid:
        print(f"Tuning {name} using GridSearchCV with {cv_folds}-fold CV...")
        search = GridSearchCV(pipe, grid, cv=cv_folds, scoring=scoring, n_jobs=-1, verbose=1)
        search.fit(X_train_reg, y_reg_train)
        print(f"  ✓ {name} best parameters: {search.best_params_}")
        print(f"  \u2713 {name} best CV Score: {search.best_score_:.4f}")
        reg_fitted[name] = search.best_estimator_
    else:
        print(f"Training {name}...")
        pipe.fit(X_train_reg, y_reg_train)
        reg_fitted[name] = pipe
        
    # Evaluate on holdout test set
    preds = reg_fitted[name].predict(X_test_reg)
    mae = mean_absolute_error(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    r2 = r2_score(y_reg_test, preds)
    reg_scores.append({"Model": name, "Test MAE": mae, "Test RMSE": rmse, "Test R²": r2})
    
    # Save model locally in Colab environment
    model_path = f"models/saved/regression/{name}.pkl"
    joblib.dump(reg_fitted[name], model_path)
    print(f"  ✓ Persisted model to: {model_path}")

reg_df = pd.DataFrame(reg_scores).sort_values("Test R²", ascending=False)
print("\n=== Regression Performance Summary ===")
print(reg_df.to_string(index=False))

# Save best model references
best_reg_name = reg_df.iloc[0]["Model"]
best_reg_pipe = reg_fitted[best_reg_name]
best_reg_path = "models/saved/best_regression_model.pkl"
joblib.dump(best_reg_pipe, best_reg_path)
print(f"\n✓ Saved overall best regression model ({best_reg_name}) to: {best_reg_path}")

## 7. Execute Classification Model Training & Tuning

In [ ]:
# Split dataset
X_train_clf, X_test_clf, y_clf_train, y_clf_test = split(engineered_df, "Burnout_Risk_Level")
preprocessor = build_preprocessor()

cv_folds = CFG["tuning"]["cv_folds"]
scoring = CFG["tuning"]["scoring_classification"]

clf_fitted = {}
clf_scores = []

for name in CFG["models"]["classification"]:
    pipe = Pipeline([("preprocessor", preprocessor), ("model", CLASSIFICATION_MODELS[name])])
    grid = CLASSIFICATION_GRIDS.get(name, {})
    
    if grid:
        print(f"Tuning {name} using GridSearchCV with {cv_folds}-fold CV...")
        search = GridSearchCV(pipe, grid, cv=cv_folds, scoring=scoring, n_jobs=-1, verbose=1)
        search.fit(X_train_clf, y_clf_train)
        print(f"  ✓ {name} best parameters: {search.best_params_}")
        print(f"  ✓ {name} best CV Score: {search.best_score_:.4f}")
        clf_fitted[name] = search.best_estimator_
    else:
        print(f"Training {name}...")
        pipe.fit(X_train_clf, y_clf_train)
        clf_fitted[name] = pipe
        
    # Evaluate on holdout test set
    preds = clf_fitted[name].predict(X_test_clf)
    acc = accuracy_score(y_clf_test, preds)
    f1 = f1_score(y_clf_test, preds, average="weighted")
    clf_scores.append({"Model": name, "Test Accuracy": acc, "Test F1": f1})
    
    # Save model locally in Colab environment
    model_path = f"models/saved/classification/{name}.pkl"
    joblib.dump(clf_fitted[name], model_path)
    print(f"  ✓ Persisted model to: {model_path}")

clf_df = pd.DataFrame(clf_scores).sort_values("Test F1", ascending=False)
print("\n=== Classification Performance Summary ===")
print(clf_df.to_string(index=False))

# Save best model references
best_clf_name = clf_df.iloc[0]["Model"]
best_clf_pipe = clf_fitted[best_clf_name]
best_clf_path = "models/saved/best_classification_model.pkl"
joblib.dump(best_clf_pipe, best_clf_path)
print(f"\n✓ Saved overall best classification model ({best_clf_name}) to: {best_clf_path}")

## 8. Save Metadata & Pack Models for Download

In [ ]:
import json
metadata = {
    "best_regression_model"     : best_reg_name,
    "best_regression_metrics"   : reg_df.iloc[0].to_dict(),
    "best_classification_model" : best_clf_name,
    "best_classification_metrics": clf_df.iloc[0].to_dict(),
    "feature_columns"           : X_train_reg.columns.tolist(),
}
with open("models/saved/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata saved ✓")

# Compress outputs to a zip archive for easy download
!zip -r models_colab.zip models/
print("\n✓ ZIP package generated: models_colab.zip")
print("Download this package from the Files side panel in Colab, extract it in your local workspace root, and run the Streamlit app!")